Có N khách hàng 1, 2, …, N cần được bảo trì mạng internet. Khách hàng i ở địa điểm i (i = 1,…,N) 

• Việc bảo trì cho khách hàng i kéo dài d(i) đơn vị thời gian (s) 

• Có K nhân viên kỹ thuật ở trụ sở công ty (điểm 0) và có thời điểm bắt đầu là việc là t0 = 0. 

• Thời gian di chuyển từ điểm i đến điểm j là t(i,j) <= 10.000

• **Lập kế hoạch phân công nhân viên thực hiện bảo trì cho các khách hàng sao cho thời gian làm việc nhiều nhất (thời gian di chuyển công thời gian bảo trì) của một nhân viên nào đó là nhỏ nhất**

A route of staff k is represented by a sequence of points r[0], r[1], r[2], . . ., r[Lk] in which r[0] = r[Lk] = 0 (the depot)

### Input

Line 1: contains N and K (1 <= N <= 1000, 1 <= K <= 100)

Line 2: contains d(1), d(2), . . ., d(N)  (1 <= d(i) <= 1000)

 Line i + 3 (i = 0, 1, 2, . . ., N): contains the ith row of the matrix t such 
 that t(i, j) <= 10000

### Output

Line 1: contains K

Line 2k (k = 1, . . ., K): contains a positive integer Lk

Line 2k+1 (k = 1, 2, . . ., K): contains  r[0], r[1], r[2], . . ., r[Lk]

### Example
```Input
5 2
60 80 70 10 90 
0 50 100 60 40 80 
50 0 50 40 20 60 
100 50 0 50 70 40 
60 40 50 0 60 20 
40 20 70 60 0 80 
80 60 40 20 80 0 

Output
2
5
0 5 3 4 0
4
0 2 1 0

# Mục tiêu: **min max(Worktime(k))** 
Trong đó: Worktime(k) = TravelTime(k) + ServericeTime(k)

=> Nhân viên nào bận nhất thì phải bận ít nhất có thể

## Greedy + local search approximation

B1: Dùng Greedy để tìm initial solution: ``PATH_CHEAPEST_ARC``. Luôn nối đến điểm gần nhất trước (heuristic xấp xỉ)

B2: Dùng Local search để cải thiện nghiệm: ``GUIDED_LOCAL_SEARCH``. (metaheuristic cho VRP)

In [1]:
from ortools.constraint_solver import pywrapcp
from ortools.constraint_solver import routing_enums_pb2

In [12]:
"""
    N: số khách hàng
    K: số nhân viên
    d: thời gian bảo trì, d[0] = 0
    t: ma trận thời gian di chuyển
"""

def solver_network_maintenance(N, K, d, t):
    manager = pywrapcp.RoutingIndexManager(
        N+1, #N khach + depot 0
        K, #so route = so nhan vien
        0 #starts: depot
    )

    routing = pywrapcp.RoutingModel(manager)

    #cost callback
    def transit_callback(from_index, to_index):
        from_note = manager.IndexToNode(from_index)
        to_node = manager.IndexToNode(to_index)

        travel = t[from_note][to_node]
        
        #chỉ note tới (to_note hay arrived node) là khách hàng thì mới có thời gian bảo trì (node di chuyển không có)
        service = 0 if to_node == 0 else d[to_node]

        return travel + service
    
    transit_idx = routing.RegisterTransitCallback(transit_callback)

    #cost cho toan route (tat ca cac)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_idx)

    #dimension để minimize max route (https://developers.google.com/optimization/routing/dimensions)
    routing.AddDimension(
        evaluator_index=transit_idx, #evaluator_index: id cua ham cost
        slack_max=0, #slack: không có thời gian đợi tại mỗi điểm (slack time i: độ trễ): đặt về 0
        capacity=10**9, #capacity: bài toán không liên quan đến carry (CVRP): đặt về vô cùng để không bị ràng buộc
        fix_start_cumul_to_zero=True, #fix_start_cumulative_to_zero: Boolean value. Nếu đúng, giá trị tích lũy của đại lượng sẽ bắt đầu từ 0 (trong bài này là thời gian sẽ tính từ 0s và tích lũy qua các node)
        name="Time" # dimension_name: string name cho dimension
    )

    time_dim = routing.GetDimensionOrDie("Time")

    #minimize max workload
    time_dim.SetGlobalSpanCostCoefficient(100)
    # span = max(route_end) - min(route_start)  : vì start = 0 => chính là minimize max workload

    #params
    search_params = pywrapcp.DefaultRoutingSearchParameters()

    #nghiệm ban đầu heuristic (greedy)
    search_params.first_solution_strategy = (routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)
    #cải thiện nghiệm metaheuristic (local search)
    search_params.local_search_metaheuristic = (routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH)

    search_params.time_limit.seconds = 10

    #solve
    solution = routing.SolveWithParameters(search_params)
    if not solution:
        print("No solution found")
        return
    
    #output đúng định dạng
    print(K)
    
    for vehicle_id in range(K):
        index = routing.Start(vehicle_id)
        route = []

        while not routing.IsEnd(index): #khi route cua nguoi vehicle_id chua ket thuc (quay ve diem 0 depot)
            route.append(manager.IndexToNode(index))
            index = solution.Value(routing.NextVar(index))

        route.append(0)

        print(len(route))
        print(*route)

In [3]:
#example huststack
N = 5
K = 2

d = [0, 60, 80, 70, 10, 90]

t = [
    [0, 50, 100, 60, 40, 80],
    [50, 0, 50, 40, 20, 60],
    [100, 50, 0, 50, 70, 40],
    [60, 40, 50, 0, 60, 20],
    [40, 20, 70, 60, 0, 80],
    [80, 60, 40, 20, 80, 0]
]

In [14]:
solver_network_maintenance(N, K, d, t)

2
5
0 2 1 4 0
4
0 3 5 0


Đối với optimization problem, không phải bài toán chỉ có một đáp án duy nhất.

Có thể tồn tại nhiều phương án phân công khác nhau cho cùng một input. Đặc biệt với thuật toán xấp xỉ